# Organizer compiler source collection28

CPU only, Internet off, official inference wheelhouse, original environment.
Copies the completed CPU19 environment definitions and collects all100 named
organizer package Python source files, each checked against the completed CPU19
byte inventory. No task/snapshot/reference/test patch, model, compiler invocation
or grader is loaded. Source collection is preparation for actual compiler review,
not validation of an agent archive or Stage1 budget semantics.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import os, sys, json, hashlib, base64, zipfile, io, time, inspect
IS_KAGGLE = Path('/kaggle/working').is_dir()
RUN_DECLARED_EXPERIMENT = True
MODE = 'cpu_controls'
WHEELHOUSE_DIR = Path('/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse')
if not IS_KAGGLE:
    print('Local definitions only: no installation, task read, inference or grader call')


In [ ]:
"""Export only artifacts produced inside a run, including per-trial checkpoints."""
import base64
import hashlib
import io
import json
import os
from pathlib import Path
import tempfile
import zipfile


def export_run(run_root: Path, *, emit: bool = True) -> dict:
    run_root = run_root.resolve(strict=True)
    artifacts, manifest = [], []
    for artifact in sorted(run_root.rglob('*')):
        resolved = artifact.resolve()
        if artifact.is_symlink() or not resolved.is_relative_to(run_root):
            raise ValueError('Run artifact is a link or escaped its directory')
        name = artifact.relative_to(run_root).as_posix()
        if name == 'evidence_bundle.zip' or name.startswith('evidence_checkpoints/'):
            continue
        if not artifact.is_file():
            continue
        if name == 'artifact_manifest.json':
            raise ValueError('artifact_manifest.json is reserved for the exported manifest')
        # Read once. A live log may grow, but its archived bytes and hash must agree.
        payload = artifact.read_bytes()
        artifacts.append((name, payload))
        manifest.append({'path': name, 'bytes': len(payload),
                         'sha256': hashlib.sha256(payload).hexdigest()})
    if sum(item['bytes'] for item in manifest) > 32 * 1024 * 1024:
        raise ValueError('Use Kaggle output download for this large evidence bundle')
    bundle = io.BytesIO()
    with zipfile.ZipFile(bundle, 'w', zipfile.ZIP_DEFLATED) as archive:
        for name, payload in artifacts:
            archive.writestr(name, payload)
        archive.writestr('artifact_manifest.json', json.dumps(manifest, indent=2) + '\n')
    bundle_bytes = bundle.getvalue()
    digest = hashlib.sha256(bundle_bytes).hexdigest()
    checkpoints = run_root / 'evidence_checkpoints'
    checkpoints.mkdir(exist_ok=True)
    checkpoint = checkpoints / (digest + '.zip')
    if checkpoint.exists():
        if checkpoint.read_bytes() != bundle_bytes:
            raise ValueError('Existing checkpoint differs from its content hash')
    else:
        with checkpoint.open('xb') as stream:
            stream.write(bundle_bytes)
            stream.flush()
            os.fsync(stream.fileno())
    temporary = None
    try:
        with tempfile.NamedTemporaryFile(dir=run_root, prefix='.evidence_bundle.',
                                         suffix='.tmp', delete=False) as stream:
            temporary = Path(stream.name)
            stream.write(bundle_bytes)
            stream.flush()
            os.fsync(stream.fileno())
        os.replace(temporary, run_root / 'evidence_bundle.zip')
    finally:
        if temporary is not None and temporary.exists():
            temporary.unlink()
    report = {'sha256': digest, 'bytes': len(bundle_bytes), 'file_count': len(manifest),
              'checkpoint_path': checkpoint.relative_to(run_root).as_posix(),
              'snapshot_is_atomic_across_all_files': False}
    if emit:
        print('NEURAL_SWE_BUNDLE_SHA256=' + report['sha256'], flush=True)
        print('NEURAL_SWE_BUNDLE_BASE64=' + base64.b64encode(bundle_bytes).decode('ascii'), flush=True)
    return report


In [ ]:
PLAN = json.loads('{"created_utc": "2026-10-05T16:07:32.740774+00:00", "scope": "installed_compiler_sources_no_tasks_model_or_grader", "expected_core_packages": {"swegemma": "0.2.7", "adk-submission": "0.2.12", "adk-eval-core": "0.1.0"}, "expected_verify_task_sha256": "98600fd6066a4dbd472490cb194b839a3a633afea9b99256cc081ea77beac0d3", "expected_organizer_python_sources": [{"distribution": "swegemma", "version": "0.2.7", "package": "swegemma", "package_relative_path": "swegemma/__init__.py", "bytes": 3857, "sha256": "355aaa3a1cf2b4b3bb5b5b6b3e72acc3481c657918411b4f18bde76415284639"}, {"distribution": "swegemma", "version": "0.2.7", "package": "swegemma", "package_relative_path": "swegemma/__main__.py", "bytes": 145, "sha256": "5cd5c92ee1a230e24c17d693c3093edf0e3e78b96c08815351ba82432a1aa4f4"}, {"distribution": "swegemma", "version": "0.2.7", "package": "swegemma", "package_relative_path": "swegemma/budget.py", "bytes": 1305, "sha256": "60bd0f73e0fcede68a3024a5e37836b0bcb0d133952044d7dc3e5a4102f2c2f9"}, {"distribution": "swegemma", "version": "0.2.7", "package": "swegemma", "package_relative_path": "swegemma/cli.py", "bytes": 5221, "sha256": "7197c5fd6b141d7df805dd350130944c84b0b1fc14ed5eeef3f52b2e26d5e30d"}, {"distribution": "swegemma", "version": "0.2.7", "package": "swegemma", "package_relative_path": "swegemma/config.py", "bytes": 7440, "sha256": "f035916218f895796c4d39f1f9ec25284152a6df57cf6de1fea4931165d2349f"}, {"distribution": "swegemma", "version": "0.2.7", "package": "swegemma", "package_relative_path": "swegemma/context.py", "bytes": 23710, "sha256": "97f06280c50df3132d1af7ef6126884b66b48887a1cf94b293af01e5907dc2a7"}, {"distribution": "swegemma", "version": "0.2.7", "package": "swegemma", "package_relative_path": "swegemma/deduplication.py", "bytes": 49745, "sha256": "f8353a8da887fc45823b9174f4578cd816b341d52c3a074964c1c2310af1573e"}, {"distribution": "swegemma", "version": "0.2.7", "package": "swegemma", "package_relative_path": "swegemma/difficulty.py", "bytes": 16760, "sha256": "0b97e060fa6a3b437024a45c42cb131526d5a48b74dd858e4fc0259301f7a6ba"}, {"distribution": "swegemma", "version": "0.2.7", "package": "swegemma", "package_relative_path": "swegemma/display.py", "bytes": 3976, "sha256": "d2dfff4ebe0ff655a240b5c1f073e85702869f618c0aa5c95cc6b54b20be47c8"}, {"distribution": "swegemma", "version": "0.2.7", "package": "swegemma", "package_relative_path": "swegemma/edit.py", "bytes": 2573, "sha256": "d34acdff4530ed211325117395cf7fa9d085f7ee84250d02acf9c8533657a956"}, {"distribution": "swegemma", "version": "0.2.7", "package": "swegemma", "package_relative_path": "swegemma/evaluate.py", "bytes": 25197, "sha256": "96a46f49267ffcf861f7f90b98efbe828d03f27553ceefce745819028589de1e"}, {"distribution": "swegemma", "version": "0.2.7", "package": "swegemma", "package_relative_path": "swegemma/graph/__init__.py", "bytes": 1372, "sha256": "9d9c6974861db6d3bea45cf2dee60ce1ba9073293138ad5157ad696ca23d4f7a"}, {"distribution": "swegemma", "version": "0.2.7", "package": "swegemma", "package_relative_path": "swegemma/graph/embedding_utils.py", "bytes": 14515, "sha256": "eb1c777ea521416f56e60ad258dc14c412c3afca1687e7244a01d59657b641c9"}, {"distribution": "swegemma", "version": "0.2.7", "package": "swegemma", "package_relative_path": "swegemma/graph/graph_utils.py", "bytes": 15651, "sha256": "e48d9f0cabe02e9ce8b1a0ab437de68a90e0a740b61a3f54ce5a95ab1f05c1cc"}, {"distribution": "swegemma", "version": "0.2.7", "package": "swegemma", "package_relative_path": "swegemma/graph/retrieval_utils.py", "bytes": 32100, "sha256": "5d18a87de66af0c773de7d4601f37c74d5fb575c9fc73763cc803a79fa74d84d"}, {"distribution": "swegemma", "version": "0.2.7", "package": "swegemma", "package_relative_path": "swegemma/harness/__init__.py", "bytes": 1079, "sha256": "ca5053d64fc35f7d457974879656e63fa7980a7b36d37a28bbe7391f1fa90fe5"}, {"distribution": "swegemma", "version": "0.2.7", "package": "swegemma", "package_relative_path": "swegemma/harness/agent_runner.py", "bytes": 35490, "sha256": "dff7d2505340633c3fe50ba7073f2609fb2b41becd69c356d9af25cc4c1f98dd"}, {"distribution": "swegemma", "version": "0.2.7", "package": "swegemma", "package_relative_path": "swegemma/harness/container_setup.py", "bytes": 40661, "sha256": "3192acdce35f2d0bd0b191186851b46c251390af16b7f139a1640e16fb1d862c"}, {"distribution": "swegemma", "version": "0.2.7", "package": "swegemma", "package_relative_path": "swegemma/harness/sample_verification.py", "bytes": 20399, "sha256": "0bbd4322534442420a4f782ffb76a5467d6a9d9bd7c32fe212269f304e05fa41"}, {"distribution": "swegemma", "version": "0.2.7", "package": "swegemma", "package_relative_path": "swegemma/harness/verification.py", "bytes": 22102, "sha256": "5602f33acb49062de8e99073199745679d86445297be9dd075ccec60c41b60dd"}, {"distribution": "swegemma", "version": "0.2.7", "package": "swegemma", "package_relative_path": "swegemma/models/__init__.py", "bytes": 793, "sha256": "34ecd27654b031cd3ede1cead44ed0e8f72bbfe6ffea24c22bfe66d493edbd97"}, {"distribution": "swegemma", "version": "0.2.7", "package": "swegemma", "package_relative_path": "swegemma/models/discovery.py", "bytes": 5681, "sha256": "70ff2d1719f71cd4c0d747fecfcf5763c1fbbb983684cc2d68b6771e3a3997ca"}, {"distribution": "swegemma", "version": "0.2.7", "package": "swegemma", "package_relative_path": "swegemma/models/registry.py", "bytes": 7374, "sha256": "69e62c69d461a11f1655285652872005aee11aa549beaec331742dc742593ca3"}, {"distribution": "swegemma", "version": "0.2.7", "package": "swegemma", "package_relative_path": "swegemma/models/task.py", "bytes": 7693, "sha256": "0b9d705ce759645bd1ee59bb28662a25a951990c024bfeb1e00a41f08350812a"}, {"distribution": "swegemma", "version": "0.2.7", "package": "swegemma", "package_relative_path": "swegemma/results.py", "bytes": 5200, "sha256": "4705de95a05ae5f01f743cbb97046fb0f47f1407f4cd1db249489b2253a7f72f"}, {"distribution": "swegemma", "version": "0.2.7", "package": "swegemma", "package_relative_path": "swegemma/sandbox/__init__.py", "bytes": 829, "sha256": "ba3725efee8f6d0fba2321851d66d4497ae3cda5863edc0d4a8ef232f1fdb775"}, {"distribution": "swegemma", "version": "0.2.7", "package": "swegemma", "package_relative_path": "swegemma/sandbox/base.py", "bytes": 10113, "sha256": "aab491aabc84105963bead16dab4d3d5745e31727ba248fd5a568d8f4e0c7dc1"}, {"distribution": "swegemma", "version": "0.2.7", "package": "swegemma", "package_relative_path": "swegemma/sandbox/docker.py", "bytes": 34531, "sha256": "cc10c287acf7c9326f622e91b98207bb9cdcf4fadf5d4aa5f5658e854051d09d"}, {"distribution": "swegemma", "version": "0.2.7", "package": "swegemma", "package_relative_path": "swegemma/sandbox/subprocess.py", "bytes": 21429, "sha256": "a26e9265d93f0387dd0a820ea451d1c530884bebca41583738f3ab2cc31df1f2"}, {"distribution": "swegemma", "version": "0.2.7", "package": "swegemma", "package_relative_path": "swegemma/submission.py", "bytes": 3577, "sha256": "d387b402223300d06eb6f4f960faf82e997098050f5317961c64e6b95365974a"}, {"distribution": "swegemma", "version": "0.2.7", "package": "swegemma", "package_relative_path": "swegemma/tools/__init__.py", "bytes": 1632, "sha256": "e19d1e81d90976fda93f346104d7f34f74018747ac81838a363de2e357358c31"}, {"distribution": "swegemma", "version": "0.2.7", "package": "swegemma", "package_relative_path": "swegemma/tools/base.py", "bytes": 3768, "sha256": "535a0e6d121549b41906685d0e440e8709c4b8b434bc31dcc82fa7740ec66281"}, {"distribution": "swegemma", "version": "0.2.7", "package": "swegemma", "package_relative_path": "swegemma/tools/execution.py", "bytes": 6055, "sha256": "f2ce1cefb41edddaeffeacbff910a54bcd0174b548efa9c6c283320e5a509912"}, {"distribution": "swegemma", "version": "0.2.7", "package": "swegemma", "package_relative_path": "swegemma/tools/graph.py", "bytes": 16411, "sha256": "7f102bf348fe799b89cd8d614afea35c5a21d8141a29859582363086ce2b1173"}, {"distribution": "swegemma", "version": "0.2.7", "package": "swegemma", "package_relative_path": "swegemma/tools/workspace.py", "bytes": 12316, "sha256": "d38ba2f68636c5e33c9f607bd9e5401dbf81d77125b966ef87c799a1615b05c6"}, {"distribution": "swegemma", "version": "0.2.7", "package": "swegemma", "package_relative_path": "swegemma/warnings.py", "bytes": 1084, "sha256": "f24d5cf69977bd88a5bb82639767ba41c12fed3bd8692969cb7c80051fbb1945"}, {"distribution": "adk-submission", "version": "0.2.12", "package": "adk_submission", "package_relative_path": "adk_submission/__init__.py", "bytes": 4369, "sha256": "ebe048f521fd5bcf46dad132f3e3dea2506df79bf471d560a70ce5b343ca387a"}, {"distribution": "adk-submission", "version": "0.2.12", "package": "adk_submission", "package_relative_path": "adk_submission/builders/__init__.py", "bytes": 637, "sha256": "ec6273c54cfb08f22777146c1636be4f13d4d00a67a17d9ae7a19e3a8f18f6c1"}, {"distribution": "adk-submission", "version": "0.2.12", "package": "adk_submission", "package_relative_path": "adk_submission/builders/llm.py", "bytes": 3658, "sha256": "97c27324ce60ac6dc466941bcc19f5482f7693af9fc02fb760a40526c84c10b5"}, {"distribution": "adk-submission", "version": "0.2.12", "package": "adk_submission", "package_relative_path": "adk_submission/builders/sub_agents.py", "bytes": 2801, "sha256": "ba6162832b1ca4e023e2848c6391242c18cbe9dadcfd560a6ebef81108c35b5a"}, {"distribution": "adk-submission", "version": "0.2.12", "package": "adk_submission", "package_relative_path": "adk_submission/builders/workflows.py", "bytes": 5565, "sha256": "d8d7ccf9a51d57daec58c7231e2a62621a3ecca269e71ee64c57062cadadf980"}, {"distribution": "adk-submission", "version": "0.2.12", "package": "adk_submission", "package_relative_path": "adk_submission/compiler.py", "bytes": 10492, "sha256": "39cc4884699faf147ca1b326664e888c1861f750fe36525b840b5f812d593eed"}, {"distribution": "adk-submission", "version": "0.2.12", "package": "adk_submission", "package_relative_path": "adk_submission/context.py", "bytes": 7403, "sha256": "3c77519415b1dd2cc97b42bcbe29d32dd85f8188ca7eba5b5fb8636bce92c908"}, {"distribution": "adk-submission", "version": "0.2.12", "package": "adk_submission", "package_relative_path": "adk_submission/discovery.py", "bytes": 31163, "sha256": "ca19fc75fb3baa42e85438b41248a06c35987cbf26ef652159b302752d0f191c"}, {"distribution": "adk-submission", "version": "0.2.12", "package": "adk_submission", "package_relative_path": "adk_submission/errors.py", "bytes": 13999, "sha256": "7e121e7c6581288788500cd1a8deceb003a74ec150d5477c37f794971a5f4818"}, {"distribution": "adk-submission", "version": "0.2.12", "package": "adk_submission", "package_relative_path": "adk_submission/limits.py", "bytes": 18965, "sha256": "afde5056f2df1fd77b53e0495712d03f60b3e246d4393617dd800d970f7482e2"}, {"distribution": "adk-submission", "version": "0.2.12", "package": "adk_submission", "package_relative_path": "adk_submission/paths.py", "bytes": 9379, "sha256": "7c4b7050141bdb05a9e6849be11f6440f2a18082a364623226c4343649b2f59f"}, {"distribution": "adk-submission", "version": "0.2.12", "package": "adk_submission", "package_relative_path": "adk_submission/registry.py", "bytes": 11173, "sha256": "c17f9770c2f3663309dd61807d2ae4e7d75bbd06b7c674847f655d218e82cf7a"}, {"distribution": "adk-submission", "version": "0.2.12", "package": "adk_submission", "package_relative_path": "adk_submission/resolvers/__init__.py", "bytes": 800, "sha256": "b88fe281c2b927a046e6485c7e0966b1808a5c8d0a0849cd85824ffb04ae53be"}, {"distribution": "adk-submission", "version": "0.2.12", "package": "adk_submission", "package_relative_path": "adk_submission/resolvers/callbacks.py", "bytes": 3643, "sha256": "332ce0732aa3eb1b555630c57683f51e59d5caee3f519371d376210e9ae33e29"}, {"distribution": "adk-submission", "version": "0.2.12", "package": "adk_submission", "package_relative_path": "adk_submission/resolvers/generation.py", "bytes": 13605, "sha256": "ba8bf237261220d46843c8d71118faa387fc07f876f072756dff44d641dd1c14"}, {"distribution": "adk-submission", "version": "0.2.12", "package": "adk_submission", "package_relative_path": "adk_submission/resolvers/models.py", "bytes": 3179, "sha256": "39070f6ae52ad8880750b51af373da8762850d3a8ecc65c6f11744afe4cd5066"}, {"distribution": "adk-submission", "version": "0.2.12", "package": "adk_submission", "package_relative_path": "adk_submission/resolvers/tools.py", "bytes": 10593, "sha256": "ae44a5ca815a05edfd763b92283d0444c77755afbd827fc35bcb87eb2633259e"}, {"distribution": "adk-submission", "version": "0.2.12", "package": "adk_submission", "package_relative_path": "adk_submission/schema.py", "bytes": 17404, "sha256": "6554de60cfa208dcb0b095f4236271795b2955adf87f3be3c40dfa3d62521c5d"}, {"distribution": "adk-submission", "version": "0.2.12", "package": "adk_submission", "package_relative_path": "adk_submission/server.py", "bytes": 35447, "sha256": "d60b3fe34dd7f6e359b003db82b68434757c74b1030862bacda5cff6164f91eb"}, {"distribution": "adk-submission", "version": "0.2.12", "package": "adk_submission", "package_relative_path": "adk_submission/yaml_loader.py", "bytes": 12618, "sha256": "01d644ef7f5acc216506245fd7e115e7096641e62e825c6873c83e148b23c69c"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/__init__.py", "bytes": 4122, "sha256": "528e02f35eaef6fa733acc0a026c80cc191d28d4a4192a740f9b7bc8a88c9211"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/budget/__init__.py", "bytes": 254, "sha256": "4a709384174fb5be5aa22c9a1385ba830d82bf03fab1798280b350311e75588e"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/budget/budget.py", "bytes": 15780, "sha256": "16de25ae326f0f8440772672a4bf7198a4ebb5586c7c3f2b31048ca896f04ab6"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/editing/__init__.py", "bytes": 266, "sha256": "6e4f2c12837a366147d582b9ccfc4f6c3fd0d117e311767e292abe8b84642397"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/editing/edit.py", "bytes": 14676, "sha256": "a78a761d5078df92606c63b3adb61991a2878bcbb80204875d241175cb532a6f"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/errors.py", "bytes": 5691, "sha256": "2a8d545695a89c0abbc8984d771df6ae31155998b8e20cb91da02d0a5b900373"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/models/__init__.py", "bytes": 246, "sha256": "1a9e1673840b8be5bcbf93767a1d130c007abf350cfc2c263e389135bba223af"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/models/base.py", "bytes": 9724, "sha256": "bdedef267b1022ffac970c7bc18c2d0a53753ce1f120a3c355b91c9b7b666a0d"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/plugins/__init__.py", "bytes": 227, "sha256": "dd53fad958a5719da6d6ab6869f59daa7c70060a257dedc77551873557b829f9"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/plugins/display_plugin.py", "bytes": 5648, "sha256": "7250793fa30985086c193fc55d5b81f4c4d115b44ad764e546a8cc1b0ac20d69"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/plugins/retry_plugin.py", "bytes": 7084, "sha256": "ba61cf201167aaf970c3131a4f24b464515509f3cacd5358de99c5ec0fb07af0"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/runner/__init__.py", "bytes": 293, "sha256": "2faaa7d8445299656396987f7b55876025b5dfaab793afa0c11027ab08984b01"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/runner/base.py", "bytes": 1198, "sha256": "fa97acbfdf8be95b9b247e525fb4fa682c8b8da156b405f5bc29f684eafb4d5f"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/runner/runner.py", "bytes": 3321, "sha256": "0c057f402eb42f0bdc4e84fd88d064287f3929284d8f345813df8dedeb7b9cc3"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/sandbox/__init__.py", "bytes": 751, "sha256": "b9f13699e2bc09fa57815bb3c6bf07c4aefb9ab5b74e80ec46625f8421b10d5a"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/sandbox/base.py", "bytes": 7694, "sha256": "da53e3bb0dfb7d2490d63195740d94560291c37001c4ebae4d12733cf4856007"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/sandbox/containers.py", "bytes": 21219, "sha256": "c36a7077901aafcd8a16ae22441dfe40d8713e36bbc8534ab2bad1cd953bf769"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/sandbox/docker_sandbox.py", "bytes": 7011, "sha256": "484b8864c2ff876f10b85f95e69e72b245e8ca5f6f30dc1020baec846212c899"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/sandbox/subprocess_sandbox.py", "bytes": 11715, "sha256": "09dc213bd71f4ad33a2a403732271efbc78bd09db2d2999817d55f4e630ac6fc"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/tracing/__init__.py", "bytes": 408, "sha256": "d1b4ed94c3c5fa97541f593d62645cc73fe60587da6d2deb742c48ac00f01096"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/tracing/atif.py", "bytes": 5302, "sha256": "266324e80f6a4fc23aa3d809f677a6e950a7bc971f8e903ce3f55dc2c0cd8f35"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/tracing/atif_converter.py", "bytes": 7430, "sha256": "1d35490d305e1676d09c11a54301488cdecc6aceb03ee3469831d3f8a1cdd992"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/tracing/markdown.py", "bytes": 7160, "sha256": "e8e07b6a71926196e09a18f08f0cc3ff6da5820bff95fcc0534fdeab083fbb6d"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/tracing/trace.py", "bytes": 23186, "sha256": "b426e17c03b420f38ee4e08c038678ff4ba66d07ea893638f4e22a9fba8a4bc5"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/ui/__init__.py", "bytes": 1571, "sha256": "3f39053c2fd360dedca776c611df55b0a4382e910f3681b0f9dbb3f08b0b459d"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/ui/dashboard.py", "bytes": 18158, "sha256": "c56edca53fe06efb29151ef140b25ac99f795b08faf0bf0632abae18ce544eec"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/ui/event_display.py", "bytes": 23833, "sha256": "4efbb0a55b5cd93cd0d8036c2d3f5f65b67072a0f5dc197e2e7b360a791f4a3e"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/ui/formatters/__init__.py", "bytes": 1170, "sha256": "2502098e77ae11a186177c980cb30560e3cd90703026a0c3dda95e191bbce288"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/ui/formatters/command.py", "bytes": 4046, "sha256": "50856b977eb86fd21f312a340e6d782101b934564a3527348e869294aa84d960"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/ui/formatters/file.py", "bytes": 5709, "sha256": "5ba0b3a5e51925431545a0f2f408725d4e2e487b934f821c635fb364b10511bf"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/ui/formatters/generic.py", "bytes": 3922, "sha256": "f458b8993b7158c3a5a73991c0adb3cb35509cd927a8ab3b14df4a515020b3be"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/ui/formatters/models.py", "bytes": 5146, "sha256": "5445d681fa4f4dc9847cda9388af8c623fd7e1e73bce15f8c42de9e34a8f84ea"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/ui/formatters/python.py", "bytes": 2119, "sha256": "3249c42b0b2d73a447913d07076623f3dbd7401b63726d9e3a3a9777ff5906e0"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/ui/formatters/registry.py", "bytes": 6171, "sha256": "b63f7260e8219d36a3c3f4ebd1001c7d729d1f106446b9ca50620f79a2e092c6"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/ui/hud.py", "bytes": 14051, "sha256": "bfc95127fd9594e5e9269071f3d7241875ab33d026316482cb157111a790757c"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/ui/presenter.py", "bytes": 7238, "sha256": "626d70427727c5e0a8ffcf37032323c3516461640694fd9be2968dbabb8a96d4"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/ui/protocol.py", "bytes": 4050, "sha256": "84d85f3e5adbc74feb7d4bedaf96131541cad5ac57012abb86f9cf4b046feb8a"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/ui/summary.py", "bytes": 592, "sha256": "3bba083cf4d57081c50abb1af49307abe586a39ea1ebf2b9891783331085b2fe"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/utils/__init__.py", "bytes": 437, "sha256": "630451edfc5b0032078ed858d971e8520f2ea776a2231aa5b7dc1a9a1958cd78"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/utils/scoring.py", "bytes": 3505, "sha256": "f6c561e79fbfea17be26c526e96d0ed11a6d57edada61ff2cd512f29eceeb69d"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/utils/utils.py", "bytes": 4562, "sha256": "df512c616a387afdb69eb5841ad8aa7915f042f9c85ab17594f03bf4a27f2df3"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/viewer/__init__.py", "bytes": 159, "sha256": "e477b6d54612929ba547af4d829ece61a74882c7d2d92c29f76f2e97c370bcf9"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/viewer/templates.py", "bytes": 9661, "sha256": "d7ec71cf42a84cb1c196ad9e7b4bef95611f6353a7d22182b7b5f147d9581b03"}, {"distribution": "adk-eval-core", "version": "0.1.0", "package": "adk_eval_core", "package_relative_path": "adk_eval_core/viewer/viewer.py", "bytes": 27504, "sha256": "16e4eebf920d0de53587c4008d1a0c4b365527940aef2f278c0cfc5fc3c14fc9"}], "completed_cpu19_source_inventory_sha256": "37f59d08d41104595da2db5bdcc4131033b40bb210f42c8e1cf019d1e6cacc26"}')
SOURCE_BINDING = json.loads('{"parent_notebook_sha256": "bd5b55cdabe47685106f596df7af911e8a1d5c1aa30f4d62a21deef2d2840895", "source_sha256": {"competition/compiler_source_collection.py": "04c829d21eec06a7bdd218b1824723d0da0ae3a0310820dba1e16b8bf089f857", "scripts/prepare_compiler_source_collection.py": "15436f9ec2e1c197cc77a58c3e5193c7e2c8957ad893e04809236db9871c3a25"}, "plan_sha256": "86709fe70a51fd524a582faa2e05c5d8c359b21b91dcc4807efae484c8ae2663", "completed_cpu19_source_inventory_sha256": "37f59d08d41104595da2db5bdcc4131033b40bb210f42c8e1cf019d1e6cacc26", "copied_parent_cells": [1, 2, 4, 5, 6, 7]}')
PLAN_BYTES_BASE64 = '...'
if IS_KAGGLE and RUN_DECLARED_EXPERIMENT:
    RUN = Path('/kaggle/working') / ('compiler_sources_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%f'))
    RUN.mkdir()
    frozen = base64.b64decode(PLAN_BYTES_BASE64, validate=True)
    if hashlib.sha256(frozen).hexdigest() != SOURCE_BINDING['plan_sha256'] or json.loads(frozen) != PLAN:
        raise ValueError('Frozen plan differs')
    (RUN / 'plan.json').write_bytes(frozen)
    (RUN / 'source_binding.json').write_text(json.dumps(SOURCE_BINDING, indent=2) + '\n', encoding='utf-8')
    export_run(RUN)


In [ ]:
"""Fail-fast, recorded CPU setup for the unchanged organizer source audit.

Definitions do not install packages. Runtime checks never filter incompatible
wheels into a superficially successful setup or establish scoring equivalence.
"""
from datetime import datetime, timezone
import glob
import hashlib
import importlib
import importlib.metadata
import json
import os
from pathlib import Path
import platform
import subprocess
import sys
import sysconfig
import traceback


REQUIRED_CPU_RUNTIME = {
    "implementation": "CPython", "python_major_minor": [3, 12],
    "system": "Linux", "machine": "x86_64",
}


def environment_file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        while chunk := stream.read(1048576):
            digest.update(chunk)
    return digest.hexdigest()


def environment_package_versions(names):
    values = {}
    for name in sorted(names):
        try:
            values[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            values[name] = None
    return values


def cpu_runtime_snapshot(package_names):
    return {
        "implementation": platform.python_implementation(),
        "python_major_minor": list(sys.version_info[:2]),
        "python_version": sys.version, "executable": sys.executable,
        "executable_realpath": str(Path(sys.executable).resolve()),
        "system": platform.system(), "machine": platform.machine(),
        "platform": platform.platform(), "libc": list(platform.libc_ver()),
        "soabi": sysconfig.get_config_var("SOABI"),
        "multiarch": sysconfig.get_config_var("MULTIARCH"),
        "cuda_visible_devices": os.environ.get("CUDA_VISIBLE_DEVICES"),
        "package_versions": environment_package_versions(package_names),
    }


def cpu_runtime_errors(runtime):
    return [key + " must be " + repr(expected) + "; observed " + repr(runtime.get(key))
            for key, expected in REQUIRED_CPU_RUNTIME.items()
            if runtime.get(key) != expected]


def normalized_organizer_wheel_name(name):
    # Retain the organizer workaround already used in frozen notebook 10.
    return name.replace("cu128", "+cu128") if "cu128" in name and "+" not in name else name


def classify_organizer_wheels(paths, supported_tags=None):
    from packaging.tags import sys_tags
    from packaging.utils import parse_wheel_filename
    supported = set(sys_tags() if supported_tags is None else supported_tags)
    rows = []
    for path in sorted(map(Path, paths), key=lambda item: item.name):
        if not path.is_file():
            raise ValueError("Missing organizer wheel: " + str(path))
        row = {"source_path": str(path), "original_name": path.name,
               "install_name": normalized_organizer_wheel_name(path.name),
               "bytes": path.stat().st_size, "sha256": environment_file_sha256(path)}
        if "cutlass" in path.name.lower():
            row.update(selected=False, compatible=None,
                       exclusion_reason="existing_organizer_cutlass_workaround")
        else:
            try:
                distribution, version, build, tags = parse_wheel_filename(row["install_name"])
                row.update(selected=True, compatible=bool(tags & supported),
                           distribution=distribution, version=str(version),
                           wheel_tags=sorted(map(str, tags)),
                           matching_runtime_tags=sorted(map(str, tags & supported)))
            except ValueError as error:
                row.update(selected=True, compatible=False, wheel_parse_error=str(error))
        rows.append(row)
    if len({row["install_name"] for row in rows}) != len(rows):
        raise ValueError("Ambiguous organizer wheel aliases")
    return rows


def remove_recorded_cutlass_hooks():
    removed = []
    for pattern in ("/usr/local/lib/python*/dist-packages/*cutlass*.pth",
                    "/usr/local/lib/python*/site-packages/*cutlass*.pth"):
        for name in sorted(glob.glob(pattern)):
            path = Path(name)
            row = {"path": str(path), "sha256": environment_file_sha256(path)}
            path.unlink()
            removed.append(row)
    return removed


def source_audit_import_check():
    import torch
    from swegemma.config import EvalConfig
    from swegemma.evaluate import Evaluator
    from swegemma.models import load_tasks
    count = torch.cuda.device_count()
    if count != 0:
        raise ValueError("Select Accelerator None: source audit requires zero CUDA devices")
    return {"torch_version": torch.__version__, "cuda_device_count": count,
            "required_organizer_imports_succeeded": True}


def prepare_source_audit_environment(wheelhouse_dir, run_root, expected_packages,
                                     checkpoint_exporter, source_binding):
    """Install the whole selected organizer wheelhouse only after compatibility.

    Always retain runtime identity and setup failures before allowing the exact
    source inventory. UI labels are recorded separately from verified runtime.
    """
    run_root = Path(run_root)
    wheelhouse_dir = Path(wheelhouse_dir)
    run_root.mkdir(parents=True, exist_ok=False)
    package_names = set(expected_packages) | {"torch", "vllm", "pip", "packaging"}
    report = {
        "scope": "CPU_source_audit_environment_setup_not_model_or_grading",
        "created_utc": datetime.now(timezone.utc).isoformat(), "status": "started",
        "required_runtime": REQUIRED_CPU_RUNTIME, "source_binding": source_binding,
        "before": cpu_runtime_snapshot(package_names), "after": None,
        "wheelhouse_path": str(wheelhouse_dir), "wheel_manifest": [],
        "selected_wheels_omitted_for_incompatibility": [],
        "cutlass_hooks_removed": [], "pip_command": None, "pip_return_code": None,
        "model_inference_executed": False, "source_inventory_executed": False,
        "tests_executed": False, "private_scoring_equivalence_established": False,
        "runtime_image_id_verified": False,
        "requested_UI_profile": "original 2026-07-01 image, Accelerator None, Internet off",
    }

    def save():
        (run_root / "environment_receipt.json").write_text(
            json.dumps(report, indent=2) + "\n", encoding="utf-8")
        print("MOBILE_SOURCE_AUDIT_ENVIRONMENT=" + json.dumps({
            "status": report["status"], "receipt": str(run_root / "environment_receipt.json"),
            "python_version": report["before"]["python_version"],
            "runtime_errors": report.get("runtime_errors", []),
            "incompatible_wheels": report.get("incompatible_wheels", []),
            "pip_return_code": report["pip_return_code"],
        }), flush=True)
        checkpoint_exporter(run_root)

    save()
    try:
        if not wheelhouse_dir.is_dir():
            raise RuntimeError("Attach the official organizer wheelhouse before setup")
        report["wheel_manifest"] = classify_organizer_wheels(wheelhouse_dir.glob("*.whl"))
        selected = [row for row in report["wheel_manifest"] if row["selected"]]
        report["runtime_errors"] = cpu_runtime_errors(report["before"])
        report["incompatible_wheels"] = [row["original_name"] for row in selected if not row["compatible"]]
        if not selected or report["runtime_errors"] or report["incompatible_wheels"]:
            report["status"] = "preflight_rejected_before_install"
            raise RuntimeError("CPU source-audit preflight rejected this environment; "
                               "use a verified Linux x86_64 CPython 3.12 worker. "
                               "Incompatible organizer wheels were retained and no install ran.")
        report["status"] = "preflight_passed"
        save()
        links = run_root / "organizer_wheel_links"
        links.mkdir()
        for row in selected:
            (links / row["install_name"]).symlink_to(Path(row["source_path"]).resolve())
        report["cutlass_hooks_removed"] = remove_recorded_cutlass_hooks()
        command = [sys.executable, "-m", "pip", "install", "-q", "--no-index",
                   "--no-deps", "--force-reinstall"] + [str(links / row["install_name"]) for row in selected]
        report.update(status="installing", pip_command=command)
        save()
        result = subprocess.run(command, capture_output=True, text=True)
        (run_root / "organizer_install_stdout.txt").write_text(result.stdout or "", encoding="utf-8")
        (run_root / "organizer_install_stderr.txt").write_text(result.stderr or "", encoding="utf-8")
        report["pip_return_code"] = result.returncode
        importlib.invalidate_caches()
        report["after"] = cpu_runtime_snapshot(package_names)
        if result.returncode != 0:
            report["status"] = "organizer_install_failed"
            raise RuntimeError("Organizer wheel installation failed; inspect retained install logs")
        observed = {name: report["after"]["package_versions"][name] for name in expected_packages}
        if observed != expected_packages:
            report["status"] = "organizer_core_identity_rejected"
            raise ValueError("Installed organizer core packages differ from the frozen source-audit plan")
        report["status"] = "checking_CPU_organizer_imports"
        save()
        report.update(source_audit_import_check())
        report["status"] = "ready_for_unchanged_source_audit"
        save()
        return report
    except Exception as error:
        if report["status"] in {"started", "preflight_passed", "installing", "checking_CPU_organizer_imports"}:
            report["status"] = "setup_exception_before_source_audit"
        report.update(error_type=type(error).__name__, error=str(error))
        (run_root / "setup_exception.txt").write_text(traceback.format_exc(), encoding="utf-8")
        save()
        raise


In [ ]:
def resolve_organizer_audit_api(evaluator_module):
    verifier = getattr(evaluator_module, "verify_task", None)
    if not callable(verifier):
        raise ValueError("Installed Evaluator module has no callable official verify_task")
    implementation = inspect.getmodule(verifier)
    namespace = getattr(verifier, "__globals__", {})
    apis = {"verify_task": verifier}
    for name in ("resolve_task_snapshot_paths", "extract_snapshot", "sandbox_start", "sandbox_exec", "sandbox_stop"):
        # verify_task's own globals are the callables used by its retained source.
        # Snapshot resolution can be a public reexport from Evaluator's module.
        function = namespace.get(name)
        if function is None and implementation is not None:
            function = getattr(implementation, name, None)
        if function is None and name == "resolve_task_snapshot_paths":
            function = getattr(evaluator_module, name, None)
        if not callable(function):
            raise ValueError("Official verifier implementation lacks callable " + name)
        apis[name] = function
    placeholder = object()
    calls = {
        "resolve_task_snapshot_paths": ((placeholder, "instance_id", "repo"), {}),
        "extract_snapshot": ((placeholder, placeholder, placeholder),
                             {"patch_path": placeholder, "base_snapshot_path": placeholder}),
        "sandbox_start": ((placeholder,), {}),
        "sandbox_exec": ((placeholder, placeholder, "command"), {}),
        "sandbox_stop": ((placeholder, placeholder), {}),
    }
    for name, (args, kwargs) in calls.items():
        inspect.signature(apis[name]).bind(*args, **kwargs)
    for name in ("sandbox_start", "sandbox_exec", "sandbox_stop"):
        if not inspect.iscoroutinefunction(apis[name]):
            raise ValueError("Official sandbox callable is not asynchronous: " + name)
    return apis


In [ ]:
"""Recorded CPU/GPU preflight for separately frozen future public comparisons."""
from datetime import datetime, timezone
import hashlib
import importlib
import inspect
import json
from pathlib import Path
import subprocess
import sys
import tempfile
import traceback



def declared_backend_check(mode, torch):
    count = torch.cuda.device_count()
    names = [torch.cuda.get_device_name(index) for index in range(count)]
    if mode == 'cpu_controls':
        if count != 0:
            raise ValueError('Select Accelerator None for fixture controls')
    elif mode == 'gpu_comparison':
        if count != 4 or any('L4' not in name for name in names):
            raise ValueError('The declared comparison requires exactly four NVIDIA L4 GPUs')
    else:
        raise ValueError('Unknown declared experiment mode')
    return {'cuda_device_count': count, 'gpu_names': names, 'torch_version': torch.__version__}


def prepare_declared_environment(wheelhouse_dir, run_root, plan, checkpoint_exporter, mode):
    run_root, wheelhouse_dir = Path(run_root), Path(wheelhouse_dir)
    receipt = run_root / 'environment_receipt.json'
    if receipt.exists():
        raise FileExistsError('Preserve the existing environment attempt')
    expected = plan['expected_core_packages']
    names = set(expected) | {'torch', 'vllm', 'pip', 'packaging'}
    report = {'created_utc': datetime.now(timezone.utc).isoformat(),
              'scope': 'declared_public_comparison_environment_not_model_or_control_results',
              'mode': mode, 'status': 'started', 'before': cpu_runtime_snapshot(names),
              'after': None, 'wheel_manifest': [], 'incompatible_wheels': [],
              'selected_wheels_omitted_for_incompatibility': [], 'pip_return_code': None,
              'plan_sha256': environment_file_sha256(run_root / 'plan.json'),
              'model_inference_executed': False, 'fixture_controls_executed': False,
              'runtime_image_id_verified': False}
    def save():
        receipt.write_text(json.dumps(report, indent=2) + '\n', encoding='utf-8')
        print('DECLARED_ENVIRONMENT=' + json.dumps({key: report.get(key) for key in
            ('mode', 'status', 'runtime_errors', 'incompatible_wheels', 'pip_return_code')}), flush=True)
        checkpoint_exporter(run_root)
    save()
    try:
        if not wheelhouse_dir.is_dir():
            raise ValueError('Attach the complete official organizer wheelhouse')
        report['runtime_errors'] = cpu_runtime_errors(report['before'])
        report['wheel_manifest'] = classify_organizer_wheels(wheelhouse_dir.glob('*.whl'))
        selected = [row for row in report['wheel_manifest'] if row['selected']]
        report['incompatible_wheels'] = [row['original_name'] for row in selected if not row['compatible']]
        if not selected or report['runtime_errors'] or report['incompatible_wheels']:
            report['status'] = 'preflight_rejected_before_install'
            raise RuntimeError('Use a compatible Linux x86_64 CPython 3.12 organizer worker; no incompatible wheels were omitted or installed')
        report['cutlass_hooks_removed'] = remove_recorded_cutlass_hooks()
        # Link staging stays outside the exported run; its wheel identities remain
        # in the receipt. It cannot collide with strict artifact link rejection.
        with tempfile.TemporaryDirectory(prefix='neural_swe_declared_wheels_') as temporary:
            links = Path(temporary)
            for row in selected:
                (links / row['install_name']).symlink_to(Path(row['source_path']).resolve())
            command = [sys.executable, '-m', 'pip', 'install', '-q', '--no-index',
                       '--no-deps', '--force-reinstall'] + [str(links / row['install_name']) for row in selected]
            report.update(status='installing', pip_command=command)
            save()
            result = subprocess.run(command, capture_output=True, text=True, timeout=1200)
            (run_root / 'organizer_install_stdout.txt').write_text(result.stdout or '', encoding='utf-8')
            (run_root / 'organizer_install_stderr.txt').write_text(result.stderr or '', encoding='utf-8')
            report['pip_return_code'] = result.returncode
        importlib.invalidate_caches()
        report['after'] = cpu_runtime_snapshot(names)
        if result.returncode != 0:
            raise RuntimeError('Organizer installation failed; retained install logs are available')
        if {name: report['after']['package_versions'][name] for name in expected} != expected:
            raise ValueError('Installed organizer core identity differs from the frozen plan')
        import torch
        from swegemma.config import EvalConfig
        from swegemma.evaluate import Evaluator
        from swegemma.models import load_tasks
        report.update(declared_backend_check(mode, torch))
        apis = resolve_organizer_audit_api(inspect.getmodule(Evaluator))
        verifier_sha = hashlib.sha256(inspect.getsource(apis['verify_task']).encode()).hexdigest()
        report['verify_task_sha256'] = verifier_sha
        if verifier_sha != plan['expected_verify_task_sha256']:
            raise ValueError('Actual organizer verifier differs from the frozen contract')
        report['status'] = 'ready_for_' + mode
        save()
        return report
    except Exception as error:
        if report['status'] != 'preflight_rejected_before_install':
            report['status'] = 'setup_exception_before_experiment'
        report.update(error_type=type(error).__name__, error=str(error))
        (run_root / 'setup_exception.txt').write_text(traceback.format_exc(), encoding='utf-8')
        save()
        raise


In [ ]:
if IS_KAGGLE and RUN_DECLARED_EXPERIMENT:
    prepare_declared_environment(WHEELHOUSE_DIR, RUN, PLAN, export_run, MODE)
else:
    print('No organizer installation executed locally')


In [ ]:
"""Collect exact named organizer source bytes; never infer or grade a task."""
from datetime import datetime, timezone
import hashlib
import importlib.metadata
import json
from pathlib import Path


def collect_compiler_sources(run_root, plan, checkpoint_exporter):
    run_root = Path(run_root)
    target = run_root / 'compiler_source_collection.json'
    if target.exists():
        raise FileExistsError('Preserve the source collection; use a new run')
    environment = json.loads((run_root / 'environment_receipt.json').read_bytes())
    if (environment.get('mode') != 'cpu_controls'
            or environment.get('status') != 'ready_for_cpu_controls'
            or environment.get('cuda_device_count') != 0):
        raise ValueError('Source collection requires the verified CPU environment')
    collected = []
    report = {'created_utc': datetime.now(timezone.utc).isoformat(),
        'scope': 'Exact installed organizer package source collection only',
        'status': 'started', 'files': collected,
        'model_inference_executed': False, 'compiler_executed': False,
        'stage1_loader_executed': False, 'grader_executed': False,
        'task_snapshot_reference_or_test_patch_read': False}
    def save():
        target.write_text(json.dumps(report, indent=2) + '\n', encoding='utf-8')
        checkpoint_exporter(run_root)
    save()
    try:
        total = 0
        for row in plan['expected_organizer_python_sources']:
            distribution = importlib.metadata.distribution(row['distribution'])
            if distribution.version != row['version']:
                raise ValueError('Organizer version changed')
            member = Path(row['package_relative_path'])
            if member.is_absolute() or '..' in member.parts or member.suffix != '.py':
                raise ValueError('Invalid frozen package source member')
            package_root = Path(distribution.locate_file(row['package'])).resolve(strict=True)
            source_path = Path(distribution.locate_file(member.as_posix()))
            if (source_path.is_symlink() or not source_path.is_file()
                    or not source_path.resolve(strict=True).is_relative_to(package_root)):
                raise ValueError('Source is a link or escaped its package root')
            payload = source_path.read_bytes()
            if len(payload) != row['bytes'] or hashlib.sha256(payload).hexdigest() != row['sha256']:
                raise ValueError('Organizer source bytes differ from completed CPU19 inventory')
            total += len(payload)
            if total > 2 * 1024 * 1024:
                raise ValueError('Source collection exceeds the frozen capacity')
            destination = run_root / 'organizer_python' / member
            destination.parent.mkdir(parents=True, exist_ok=True)
            destination.write_bytes(payload)
            collected.append({**row, 'retained_file': destination.relative_to(run_root).as_posix()})
        report.update(status='completed', file_count=len(collected), source_bytes=total,
                      every_source_matches_completed_cpu19=True)
        save()
        return report
    except Exception as error:
        report.update(status='failed', error_type=type(error).__name__, error=str(error))
        save()
        raise


In [ ]:
if IS_KAGGLE and RUN_DECLARED_EXPERIMENT:
    result = collect_compiler_sources(RUN, PLAN, export_run)
    print('COMPILER_SOURCE_COLLECTION=' + json.dumps(result))
    export_run(RUN)
else:
    print('No source collection, model, compiler or grader executed locally')
